# FACULTY SOLUTION — Week 7: Competitive response — two market structures
Computes every answer from the canonical CSVs and asserts it against the golden fixtures.

In [ ]:
import pandas as pd, numpy as np, math, json, itertools
d = {s: pd.read_csv(f'../data/{s}.csv') for s in ['clusters', 'rival_move', 'capacity_payoffs', 'window3_params', 'cohort_states', 'worked_example_markets']}

In [ ]:

def compute(d):
    cl = d['clusters'].set_index('cluster'); p = {x.parameter: x.value for x in d['rival_move'].itertuples()}
    pay = d['capacity_payoffs'].set_index('cell_id')['payoff_musd']; w = {x.parameter: x.value for x in d['window3_params'].itertuples()}
    per_gal_value = p['fuel_margin'] + p['nonfuel_per_fill'] / p['gallons_per_fill']
    r = {}
    for c in cl.index:
        r['at_risk_pct_' + c] = cl.loc[c, 'elasticity'] * (p['street_cut'] / p['pump_base']) * 100
        r['match_cost_musd_' + c] = p['street_cut'] * cl.loc[c, 'annual_volume_mgal']
        r['ignore_cost_musd_' + c] = per_gal_value * cl.loc[c, 'annual_volume_mgal'] * abs(r['at_risk_pct_' + c]) / 100
        r['retail_decision_' + c] = 'match' if r['ignore_cost_musd_' + c] > r['match_cost_musd_' + c] else 'ignore'
    q = p['rival_build_probability']
    r['ev_hold_musd'] = q * pay['hold_builds'] + (1 - q) * pay['hold_bluffs']
    r['ev_match_musd'] = q * pay['match_builds'] + (1 - q) * pay['match_bluffs']
    r['breakeven_build_probability'] = (pay['match_bluffs'] - pay['hold_bluffs']) / ((pay['match_bluffs'] - pay['hold_bluffs']) + (pay['hold_builds'] - pay['match_builds']))
    r['capacity_decision'] = 'hold' if r['ev_hold_musd'] >= r['ev_match_musd'] else 'match'
    for s in d['cohort_states'].itertuples():
        r['nonfuel_' + s.state] = w['base_nonfuel'] - w['slope'] * (s.aggression - w['pivot'])
        r['nonfuel_change_pct_' + s.state] = (r['nonfuel_' + s.state] / w['base_nonfuel'] - 1) * 100
    return r


In [ ]:
fx = json.load(open('../fixtures/week7_golden.json'))
r = compute(d)
bad = []
for k, v in fx['results'].items():
    a = r[k]
    ok = (str(a) == str(v)) if isinstance(v, str) else math.isclose(float(a), float(v), rel_tol=1e-3, abs_tol=1e-5)
    print(('OK  ' if ok else 'DIFF'), f'{k:36s}', a)
    if not ok: bad.append(k)
assert not bad, bad
print('FACULTY NOTEBOOK REPRODUCES GOLDEN FIXTURES: PASS')